

## When and when not to apply SSS

The SSS algorithm serves to suppress external noise sources; however, there are also some disadvantages. One issue is that SSS dramatically reduces the rank of the data. This is important to take into account when performing source modeling when the covarience matrix is of importance, e.g. beamforming approaches. A second issue is that broad-band noise on the data might increase. This is not a problem per se when investigating event-related fields and modulations of oscillations below 30 Hz. However, the SSS-related broad-band noise increase might present a problem when performing multi-variate pattern analysis (see Bezudnova et al., in press). When planning to perform multi-variate pattern or representational pattern analysis, it is recommended to use signal space projection or homogeneous field correction rather than SSS. 

# Applying the MaxFilter after identifying bad sensors
The aim of this section is to demonstrate the application of the 'Maxwell filtering' also referred to as signal-space separation (SSS). The implementation termed MaxFilter serves to reduce artifacts from both environmental sources and sensor noise. The procedure also performs some fine-tuning and reduces the cross-talk between co-located sensors. Prior to applying the MaxFilter algorithm it is advisable to remove 'bad' sensors, i.e. sensors that are flat or have excessive artifacts. If this is not done, it might result in the artifacts from faulty sensors being distributed to other sensors.

Specifically, the SSS is a technique derived from the physics of electromagnetism defined by the Maxwell equations (Taulu and Kajola 2005; Taulu and Simola 2006). The algorithm separates the measured signal into components attributable to sources respectively inside and outside the MEG sensor array. As the internal and external components are linearly independent, it is possible to suppress the contribution from the external components and thus reduce environmental noise sources. The procedure also omits the higher-order components dominated by  noise. 

# Why is it necessary to use calibration files?
Inside MEG helmet, sensors are organized as triplets (one magnetometer and two orthogonal planar gradiometers).However, physical manufacturing tolerances mean coils are never perfectly flat, perfectly orthogonal, or identical in sensitivity.

Maxwell filtering projects data onto spherical harmonics. If sensor coordinates or gain calibrations are slightly off, environmental noise will project into brain components, creating phantom brain activity.


# Why crosstalk files?
Because the magnetometer and two gradiometers sit on the exact same silicon chip chip carrier (~1mm apart), their inductive loops couple to one another. A large magnetic flux in one coil induces an eddy current and parasitic flux in its neighbor.

It subtracts parasitic sensor-to-sensor leakage before Maxwell decomposition begins.


# Finding noisy and flat sensors and labelling them as bad- how and why?
mne.preprocessing.find_bad_channels_maxwell function divides the recording into sliding time windows (default 5 seconds) and performs two statistical tests- flat sensors and noisy sensors detection

Why it is critical before SSS? If you leave a noisy sensor in the dataset, Maxwell filtering will attempt to explain its localized noise using spherical harmonics, which will leak and smear the artifact across all neighboring healthy sensors. Marking it bad excludes it from the fit.

In [ ]:
import os #access to operating file
import gc # garbage collector interface for RAM relocation so that VS code doesnt freeze
import mne # main library 

# CONFIGURATION
# hierarchical STRING variables following the BIDS-like naming standard
data_dir = r'C:\Users\manju\Desktop\Academia\26_iitroorkee_a\data'
subject = 'CA103'
session = '1'
task = 'dur'
num_runs = 5  # Automatically processes 01, 02, 03, 04, 05; sets the number of runs to be looped 

# Directories
raw_data_dir = os.path.join(data_dir, f'sub-{subject}', f'ses-{session}', 'meg') # points at where to find raw data is 
deriv_dir = os.path.join(data_dir, f'sub-{subject}', f'ses-{session}', 'derivatives') # points at where to save the processed data
os.makedirs(deriv_dir, exist_ok=True) # Creates the derivatives folder on your hard drive if it does not already exist. 
# exist_ok=True prevents Python from throwing an error if the folder is already present.


# Hardware Calibration & Crosstalk files
calibration_file = os.path.join(raw_data_dir, f'sub-{subject}_ses-{session}_acq-calibration_meg.dat')
# .dat file accurate 3D sensor positions, individual channel gain adjustment and coil
crosstalk_file = os.path.join(raw_data_dir, f'sub-{subject}_ses-{session}_acq-crosstalk_meg.fif')
# .fif crosstalk file contains a mutual inductance matrix 

print("Starting Maxwell Filtering batch...")
print(f"Calibration File: {calibration_file}")
print(f"Crosstalk File:   {crosstalk_file}")
print("=" * 70)

# AUTOMATED BATCH LOOP
for run in range(1, num_runs + 1):
    run_str = f"{run:02d}"
    
    in_file = os.path.join(raw_data_dir, f"sub-{subject}_ses-{session}_task-{task}_run-{run_str}_meg.fif")
    out_file = os.path.join(deriv_dir, f"sub-{subject}_ses-{session}_task-{task}_run-{run_str}_raw_sss.fif")
    
    print(f"\n>>> [Run {run_str}/{num_runs:02d}] Loading: {os.path.basename(in_file)}")
    
    # 1. Load run
    raw = mne.io.read_raw_fif(in_file, preload=True, verbose=False)
    # Reads EOG/triggers and MEG signals directly into active RAM as a continuous NumPy arrays (NofChannels x NofSamples) for faster processing.
    raw.info['bads'] = []
    # resets the bad channel list to an empty list for each run, ensuring that any previously identified bad channels do not carry over to the next run.
    
    # 2. Automated Bad Channel Detection (Noisy & Flat Sensors)
    raw_copy = raw.copy() # Duplicates dataset in memory so filtering operations during bad-channel detection do not modify the original data.
    # flat and noisy sensors detected
    noisy_chs, flat_chs, _ = mne.preprocessing.find_bad_channels_maxwell(
        raw_copy,
        cross_talk=crosstalk_file,
        calibration=calibration_file,
        return_scores=True,
        verbose=False
    )
    bads = noisy_chs + flat_chs # now they are all labeled as bad channels and stored in the 'bads' list
    raw.info['bads'].extend(bads)
    print(f"    Bad channels identified ({len(bads)}): {bads}")
    
    # 3. Standardize Magnetometer Coil Types (FLUX Requirement)
    raw.fix_mag_coil_types() # if coil type definitions do not match the calibration file geometry; new and old ones are different apparently
    
    # 4. Apply Maxwell Filter (Signal Space Separation)- noise suppression, bad channels reconstruction 
    print("    Applying Signal Space Separation (SSS)...")
    raw_sss = mne.preprocessing.maxwell_filter(
        raw,
        cross_talk=crosstalk_file,
        calibration=calibration_file,
        verbose=False
    )
    
    # 5. Save to derivatives
    print(f"    Saving SSS output to: {os.path.basename(out_file)}")
    raw_sss.save(out_file, overwrite=True)
    
    # 6. Free RAM so the computer doesn't slow down
    del raw, raw_copy, raw_sss
    gc.collect()

print("\n" + "=" * 70)
print("COMPLETED: All 5 runs have been Maxwell-filtered and saved to derivatives!")

Starting Maxwell Filtering batch...
Calibration File: C:\Users\manju\Desktop\Academia\26_iitroorkee_a\data\sub-CA103\ses-1\meg\sub-CA103_ses-1_acq-calibration_meg.dat
Crosstalk File:   C:\Users\manju\Desktop\Academia\26_iitroorkee_a\data\sub-CA103\ses-1\meg\sub-CA103_ses-1_acq-crosstalk_meg.fif

>>> [Run 01/05] Loading: sub-CA103_ses-1_task-dur_run-01_meg.fif
    Bad channels identified (1): ['MEG2433']
102 of 102 magnetometer types replaced with T3.
    Applying Signal Space Separation (SSS)...
    Saving SSS output to: sub-CA103_ses-1_task-dur_run-01_raw_sss.fif
Overwriting existing file.
Writing C:\Users\manju\Desktop\Academia\26_iitroorkee_a\data\sub-CA103\ses-1\derivatives\sub-CA103_ses-1_task-dur_run-01_raw_sss.fif
Overwriting existing file.
Closing C:\Users\manju\Desktop\Academia\26_iitroorkee_a\data\sub-CA103\ses-1\derivatives\sub-CA103_ses-1_task-dur_run-01_raw_sss.fif
[done]

>>> [Run 02/05] Loading: sub-CA103_ses-1_task-dur_run-02_meg.fif
    Bad channels identified (1): ['M